# Cardiovascular Disease Prediction - Model Training Pipeline



## Step 1: Import Required Libraries

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

## Step 2: Load cardio_train Dataset

In [ ]:
# Locate dataset (handles execution inside backend/ or from workspace root)
csv_path = "cardio_train.csv" if os.path.exists("cardio_train.csv") else os.path.join("backend", "cardio_train.csv")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Dataset not found at {csv_path}")

with open(csv_path, "r", encoding="utf-8") as f:
    first_line = f.readline()
sep = ";" if ";" in first_line else ","

df = pd.read_csv(csv_path, sep=sep)
df.head()

## Step 3: Data Cleaning & Outlier Removal

In [ ]:
# Drop ID column if present
if "id" in df.columns:
    df = df.drop(columns=["id"])

# Remove duplicate rows
duplicates = df.duplicated().sum()
if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)

# Filter physiological outliers for blood pressure, height, and weight
initial_count = len(df)
df = df[(df["ap_hi"] >= 50) & (df["ap_hi"] <= 250)]
df = df[(df["ap_lo"] >= 40) & (df["ap_lo"] <= 180)]
df = df[df["ap_hi"] >= df["ap_lo"]]
df = df[(df["height"] >= 100) & (df["height"] <= 240)]
df = df[(df["weight"] >= 30) & (df["weight"] <= 220)]


## Step 4: Feature Engineering & Target Separation

In [ ]:
# Convert age from days to integer years
df["age"] = (df["age"] / 365.25).round().astype(int)

# 11 features matching cardio_train and the FastAPI schema
feature_names = [
    "age",
    "gender",
    "height",
    "weight",
    "ap_hi",
    "ap_lo",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active"
]
X = df[feature_names]
y = df["cardio"]


## Step 5: Train-Test Split (80% Train, 20% Test Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


## Step 6: Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Step 7: Model Training & Comparison

In [ ]:
candidate_models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, random_state=42),
        True
    ),
    "Decision Tree": (
        DecisionTreeClassifier(max_depth=6, random_state=42),
        False
    ),
    "K-Nearest Neighbors": (
        KNeighborsClassifier(n_neighbors=5),
        True
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
        True
    )
}

model_comparison = []
trained_models = {}

for name, (model, needs_scale) in candidate_models.items():
    train_features = X_train_scaled if needs_scale else X_train
    test_features = X_test_scaled if needs_scale else X_test

    model.fit(train_features, y_train)
    trained_models[name] = (model, needs_scale)

    y_pred = model.predict(test_features)
    y_proba = model.predict_proba(test_features)[:, 1]

    acc = round(accuracy_score(y_test, y_pred) * 100, 2)
    prec = round(precision_score(y_test, y_pred) * 100, 2)
    rec = round(recall_score(y_test, y_pred) * 100, 2)
    f1 = round(f1_score(y_test, y_pred) * 100, 2)
    roc_auc = round(roc_auc_score(y_test, y_proba), 4)

    print(f"  -> {name}: Acc={acc}%, Prec={prec}%, Rec={rec}%, F1={f1}%, ROC-AUC={roc_auc}")
    model_comparison.append({
        "model_name": name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "roc_auc": roc_auc
    })

comparison_df = pd.DataFrame(model_comparison)
comparison_df

## Step 8: Select Best Model (Random Forest) & Detailed Evaluation

In [ ]:
best_model_name = "Random Forest"
best_model, _ = trained_models[best_model_name]
best_pred = best_model.predict(X_test_scaled)
best_proba = best_model.predict_proba(X_test_scaled)[:, 1]

cm = confusion_matrix(y_test, best_pred).tolist()

best_metrics = {
    "selected_model": best_model_name,
    "accuracy": round(float(accuracy_score(y_test, best_pred)) * 100, 2),
    "precision": round(float(precision_score(y_test, best_pred)) * 100, 2),
    "recall": round(float(recall_score(y_test, best_pred)) * 100, 2),
    "f1_score": round(float(f1_score(y_test, best_pred)) * 100, 2),
    "roc_auc": round(float(roc_auc_score(y_test, best_proba)), 4),
    "confusion_matrix": {
        "true_negatives": int(cm[0][0]),
        "false_positives": int(cm[0][1]),
        "false_negatives": int(cm[1][0]),
        "true_positives": int(cm[1][1]),
        "matrix": cm
    },
    "test_samples": len(y_test),
    "train_samples": len(y_train),
    "feature_names": feature_names,
    "model_comparison": model_comparison
}

importance = best_model.feature_importances_
feature_importance_list = []
for feat, imp in zip(feature_names, importance):
    feature_importance_list.append({
        "feature": feat,
        "importance": round(float(imp) * 100, 2)
    })
best_metrics["feature_importances"] = feature_importance_list


## Step 9: Save Production Artifacts
Export `model.pkl`, `scaler.pkl`, and `metrics.json` into `backend/` for FastAPI backend inference.

In [ ]:
# Export artifacts to current directory or backend/
output_dir = "." if os.path.exists("cardio_train.csv") else "backend"

model_out = os.path.join(output_dir, "model.pkl")
scaler_out = os.path.join(output_dir, "scaler.pkl")
metrics_out = os.path.join(output_dir, "metrics.json")

joblib.dump(best_model, model_out)
joblib.dump(scaler, scaler_out)
with open(metrics_out, "w", encoding="utf-8") as f:
    json.dump(best_metrics, f, indent=4)
